In [ ]:
import os
import glob
import numpy as np
import flammkuchen as fl
import tifffile
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.signal import convolve

In [ ]:
def create_gcamp6s_kernel(tau_rise=0.07, tau_decay=1.5, duration=5.0, sampling_rate=3.0):
    """
    Create a GCaMP6s calcium indicator kernel.
    
    Parameters:
    -----------
    tau_rise : float
        Time constant for the rising phase (in seconds)
    tau_decay : float
        Time constant for the decaying phase (in seconds)
    duration : float
        Total duration of the kernel (in seconds)
    sampling_rate : float
        Sampling rate of the data (Hz)
    
    Returns:
    --------
    numpy.ndarray
        Normalized calcium kernel
    """
    # Create time vector
    t = np.arange(0, duration, 1.0/sampling_rate)
    
    # Create normalized kernel (double exponential function)
    kernel = (1 - np.exp(-t/tau_rise)) * np.exp(-t/tau_decay)
    
    # Normalize to area = 1
    kernel = kernel / np.sum(kernel)
    
    return kernel

def convolve_regressors(regressors, kernel):
    """
    Convolve regressors with the calcium kernel.
    
    Parameters:
    -----------
    regressors : dict
        Dictionary containing 'left_regressor' and 'right_regressor'
    kernel : numpy.ndarray
        Calcium kernel
    
    Returns:
    --------
    dict
        Dictionary containing convolved regressors
    """
    # Convolve each regressor with the kernel
    left_convolved = convolve(regressors['left_regressor'], kernel, mode='full')
    right_convolved = convolve(regressors['right_regressor'], kernel, mode='full')
    
    # Trim to the original length
    orig_len = len(regressors['left_regressor'])
    left_convolved = left_convolved[:orig_len]
    right_convolved = right_convolved[:orig_len]
    
    # Create output dictionary
    convolved_regressors = {
        'left_regressor': left_convolved,
        'right_convolved': right_convolved,
        'kernel': kernel
    }
    
    return convolved_regressors

def calculate_correlation_maps(dff_data, left_regressor, right_regressor):
    """
    Calculate correlation maps between ΔF/F and motion regressors.
    
    Parameters:
    -----------
    dff_data : numpy.ndarray
        3D array of ΔF/F data (t, y, x)
    left_regressor : numpy.ndarray
        Convolved left motion regressor (1D)
    right_regressor : numpy.ndarray
        Convolved right motion regressor (1D)
    
    Returns:
    --------
    tuple
        (left_correlation_map, right_correlation_map)
    """
    # Get data dimensions
    t, y, x = dff_data.shape
    
    # Initialize correlation maps
    left_corr_map = np.zeros((y, x))
    right_corr_map = np.zeros((y, x))
    
    # Calculate correlation for each pixel
    for i in range(y):
        for j in range(x):
            pixel_trace = dff_data[:, i, j]
            
            # Calculate correlation with left regressor
            left_corr = np.corrcoef(pixel_trace, left_regressor[:t])[0, 1]
            left_corr_map[i, j] = left_corr
            
            # Calculate correlation with right regressor
            right_corr = np.corrcoef(pixel_trace, right_regressor[:t])[0, 1]
            right_corr_map[i, j] = right_corr
    
    # Handle NaN values (can occur if a pixel has no variance)
    left_corr_map = np.nan_to_num(left_corr_map)
    right_corr_map = np.nan_to_num(right_corr_map)
    
    return left_corr_map, right_corr_map

def process_session_correlation(session_folder, smooth_maps=True, sigma=1.0):
    """
    Process a single session to generate correlation maps.
    
    Parameters:
    -----------
    session_folder : str
        Path to the session folder
    smooth_maps : bool
        Whether to apply spatial smoothing to the correlation maps
    sigma : float
        Standard deviation for Gaussian smoothing
    """
    session_name = os.path.basename(session_folder)
    print(f"\nProcessing session: {session_name}")
    
    # Paths
    regressor_path = os.path.join(session_folder, "motion_regressors.h5")
    dff_path = os.path.join(session_folder, "dff_data.tif")
    output_path = os.path.join(session_folder, "correlation_maps.h5")
    
    # Check if files exist
    if not os.path.exists(regressor_path):
        print(f"  No regressor file found at {regressor_path}")
        return
    
    if not os.path.exists(dff_path):
        print(f"  No ΔF/F file found at {dff_path}")
        return
    
    # Load regressors
    try:
        print(f"  Loading motion regressors from {os.path.basename(regressor_path)}")
        regressors = fl.load(regressor_path)
        
        # Create GCaMP6s kernel
        print("  Creating GCaMP6s kernel")
        kernel = create_gcamp6s_kernel(sampling_rate=3.0)  # Assuming 3 Hz imaging
        
        # Convolve regressors with kernel
        print("  Convolving regressors with GCaMP6s kernel")
        convolved_regressors = convolve_regressors(regressors, kernel)
        
        # Get convolved regressors
        left_reg_conv = convolved_regressors['left_regressor']
        right_reg_conv = convolved_regressors['right_convolved']
        
    except Exception as e:
        print(f"  Error processing regressors: {str(e)}")
        return
    
    # Load ΔF/F data
    try:
        print(f"  Loading ΔF/F data from {os.path.basename(dff_path)}")
        dff_data = tifffile.imread(dff_path)
        
        # Check if length matches
        if len(left_reg_conv) != dff_data.shape[0]:
            print(f"  Warning: Regressor length ({len(left_reg_conv)}) doesn't match ΔF/F length ({dff_data.shape[0]})")
            # Truncate to shorter length
            min_len = min(len(left_reg_conv), dff_data.shape[0])
            left_reg_conv = left_reg_conv[:min_len]
            right_reg_conv = right_reg_conv[:min_len]
            dff_data = dff_data[:min_len]
            print(f"  Truncated to length {min_len}")
    
    except Exception as e:
        print(f"  Error loading ΔF/F data: {str(e)}")
        return
    
    # Calculate correlation maps
    print("  Calculating correlation maps")
    try:
        left_corr_map, right_corr_map = calculate_correlation_maps(
            dff_data, left_reg_conv, right_reg_conv
        )
        
        # Apply spatial smoothing if requested
        if smooth_maps:
            print("  Applying spatial smoothing")
            left_corr_map = gaussian_filter(left_corr_map, sigma=sigma)
            right_corr_map = gaussian_filter(right_corr_map, sigma=sigma)
        
        # Create direction selectivity map (right - left)
        direction_map = right_corr_map - left_corr_map
        
        # Create visualization
        visualize_correlation_maps(left_corr_map, right_corr_map, direction_map, 
                                  os.path.join(session_folder, "correlation_maps.png"))
        
        # Save correlation maps
        print(f"  Saving correlation maps to {os.path.basename(output_path)}")
        fl.save(output_path, {
            'left_correlation': left_corr_map,
            'right_correlation': right_corr_map,
            'direction_selectivity': direction_map,
            'metadata': {
                'smoothed': smooth_maps,
                'smoothing_sigma': sigma if smooth_maps else 0
            }
        })
        
        # Save as TIFF files for easy viewing
        tifffile.imwrite(os.path.join(session_folder, "left_correlation.tif"), 
                         left_corr_map.astype(np.float32))
        tifffile.imwrite(os.path.join(session_folder, "right_correlation.tif"), 
                         right_corr_map.astype(np.float32))
        tifffile.imwrite(os.path.join(session_folder, "direction_selectivity.tif"), 
                         direction_map.astype(np.float32))
        
        print("  Correlation maps processing complete")
        
    except Exception as e:
        print(f"  Error calculating correlation maps: {str(e)}")
        return

def visualize_correlation_maps(left_map, right_map, direction_map, output_path):
    """
    Create visualization of correlation maps.
    
    Parameters:
    -----------
    left_map : numpy.ndarray
        Left correlation map
    right_map : numpy.ndarray
        Right correlation map
    direction_map : numpy.ndarray
        Direction selectivity map
    output_path : str
        Path to save the visualization
    """
    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Determine correlation range for consistent color scaling
    vmax = 1#max(np.max(np.abs(left_map)), np.max(np.abs(right_map)))
    vmax = 1#min(vmax, 0.8)  # Cap at 0.8 for better visibility
    
    # Plot left correlation map
    im0 = axes[0].imshow(left_map, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[0].set_title("Left Motion Correlation")
    axes[0].axis('off')
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    
    # Plot right correlation map
    im1 = axes[1].imshow(right_map, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[1].set_title("Right Motion Correlation")
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    
    # Plot direction selectivity map
    im2 = axes[2].imshow(direction_map, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[2].set_title("Direction Selectivity (Right - Left)")
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()

def visualize_gcamp_kernel(kernel, sampling_rate=3.0, output_path="gcamp_kernel.png"):
    """
    Visualize the GCaMP kernel.
    
    Parameters:
    -----------
    kernel : numpy.ndarray
        GCaMP kernel
    sampling_rate : float
        Sampling rate (Hz)
    output_path : str
        Path to save the visualization
    """
    t = np.arange(len(kernel)) / sampling_rate
    
    plt.figure(figsize=(10, 4))
    plt.plot(t, kernel, 'b-', linewidth=2)
    plt.fill_between(t, 0, kernel, alpha=0.3, color='blue')
    plt.xlabel('Time (s)')
    plt.ylabel('Kernel Amplitude')
    plt.title('GCaMP6s Kernel')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

def process_all_sessions_correlation(base_dir, session_pattern="00*", smooth_maps=True, sigma=1.0):
    """
    Process all session folders to generate correlation maps.
    
    Parameters:
    -----------
    base_dir : str
        Base directory containing session folders
    session_pattern : str
        Pattern to match session folders (e.g., "00*")
    smooth_maps : bool
        Whether to apply spatial smoothing to the correlation maps
    sigma : float
        Standard deviation for Gaussian smoothing
    """
    # Create and visualize the GCaMP kernel once
    kernel = create_gcamp6s_kernel(sampling_rate=3.0)
    visualize_gcamp_kernel(kernel, output_path=os.path.join(base_dir, "gcamp_kernel.png"))
    
    # Find all session folders
    session_folders = sorted(glob.glob(os.path.join(base_dir, session_pattern)))
    
    if not session_folders:
        print(f"No session folders found matching pattern '{session_pattern}' in {base_dir}")
        return
    
    print(f"Found {len(session_folders)} session folders")
    
    # Process each session
    for session_folder in session_folders:
        try:
            process_session_correlation(session_folder, smooth_maps, sigma)
        except Exception as e:
            print(f"Error processing session {os.path.basename(session_folder)}: {str(e)}")
            continue

In [ ]:
from glob import glob

In [ ]:
master = Path(r"Z:\Hagar\main\e0020 imaging")

fish_list = list(master.glob("*_v41*"))
fish = fish_list[0]
print(fish)
num_fish = len(fish_list)

In [ ]:
import glob

In [ ]:
for fish in fish_list[0:]:
    base_dir = str(fish / 'suite2p')
    
    # Process all sessions
    process_all_sessions_correlation(
        base_dir,
        session_pattern="000*",  # Pattern for session folders
        smooth_maps=False,        # Apply spatial smoothing to correlation maps
        sigma=1.0                # Spatial smoothing parameter
    )